## Initialization

In [1]:
import re, os, json
import jsonschema

In [2]:
import UWjson.UWjsonL0 as L0

In [3]:
def filter_dict(sup_dict, key_cond):
    return {k:sup_dict[k] for k in sup_dict.keys() if key_cond(k)}

In [4]:
def search_value(struct, value, loc=None):

    out = []
    
    if loc is None:
        loc = []
    
    if isinstance(struct, dict):
        for k, v in struct.items():
            out.extend(search_value(v, value, loc + [k]))

    elif isinstance(struct, list):
        for i, v in enumerate(struct):
            out.extend(search_value(v, value, loc + [i]))

    else:
        if (value is None) and (struct is None):
            out.append(loc)
        elif (struct == value):
            out.append(loc)

    return out

## Decoding WMO ID map and AOML ID map

In [5]:
with open("examples/input/OcrMap.csv", "r") as infile:
    OCR_map = L0.parse_OCR_map(infile)

In [6]:
with open("examples/input/AomlIdMap", "r") as infile:
    AOML_map_txt = infile.readlines()

AOML_map = L0.parse_AOML_ID_map(AOML_map_txt)

Line 424: WARNING: non-integer value Apf ID '?'
Line 939: WARNING: multiple occurrence of Apf ID 7983
Line 1299: WARNING: multiple occurrence of Apf ID 9125
Line 1398: WARNING: multiple occurrence of Apf ID 11012
Line 1427: WARNING: multiple occurrence of Apf ID 9630
Line 1789: WARNING: multiple occurrence of Apf ID 17764
Line 1790: WARNING: multiple occurrence of Apf ID 17429
Line 1979: WARNING: multiple occurrence of Apf ID 18101
Line 2346: WARNING: multiple occurrence of Apf ID 64
Line 2347: WARNING: multiple occurrence of Apf ID 65
Line 2467: WARNING: multiple occurrence of Apf ID 1681


In [7]:
with open("examples/input/WmoIdMap", "r") as infile:
    WMO_map_txt = infile.readlines()

WMO_map = L0.parse_WMO_ID_map(WMO_map_txt)

Line 1183: WARNING: multiple occurrence of Apf ID 116
Line 1331: WARNING: non-integer value WMO ID '$1900162'
Line 3047: WARNING: multiple occurrence of Apf ID 1205
Line 3060: WARNING: multiple occurrence of Apf ID 18101
Line 3178: WARNING: multiple occurrence of Apf ID 1342


## Core CSD float 23031 test (one profile)

Note: profile 17 and 28 have duplicated engineering data sections; profile 24 is duplicated in its entirety

In [8]:
float_id = 23031
cycle_id = 17

msg = L0.msg_tokenizer(f"examples/input/{float_id}/{float_id}.{cycle_id:03}")
log1, cuts1 = L0.log_tokenizer(f"examples/input/{float_id}/{float_id}.{cycle_id:03}", fill_value=-999)
log2, cuts2 = L0.log_tokenizer(f"examples/input/{float_id}/{float_id}.{cycle_id+1:03}", fill_value=-999)
log = log1[cuts1[0]:] + log2[:cuts2[0]]

[MSG] More than 1 "<EOT>" found


In [9]:
out = L0.L0_parser(msg, log, None, None, None, AOML_map, WMO_map, None, consume=True)

In [10]:
with open("UWschema/UW_schema_L0.json", "r") as infile:
    schema = json.loads(infile.read())

In [11]:
jsonschema.validate(out, schema)

In [12]:
folder_string = f"{AOML_map[float_id]:05}"
file_string = f"{AOML_map[float_id]:05}_{float_id:06}_L0_{cycle_id:04}"
outpath = f"examples/output/{folder_string}/{file_string}.json"

if not os.path.isdir(f"examples/output/{folder_string}"):
    os.mkdir(f"examples/output/{folder_string}")

with open(outpath, "w") as outfile:
    json.dump(out, outfile, indent=2)

## Core CSD float 23031 test (loop over profiles)

Note: profile 17 and 28 have duplicated engineering data sections; profile 24 is duplicated in its entirety

In [13]:
with open("UWschema/UW_schema_L0.json", "r") as infile:
    schema = json.loads(infile.read())

In [14]:
float_id = 23031

folder_string = f"{AOML_map[float_id]:05}"
if not os.path.isdir(f"examples/output/{folder_string}"):
    os.mkdir(f"examples/output/{folder_string}")

for _i in range(40):
    file = f"examples/input/{float_id}/{float_id}.{_i:03}"
    file2 = f"examples/input/{float_id}/{float_id}.{_i+1:03}"
    print(file)
    msg = L0.msg_tokenizer(file)
    log1, cuts1 = L0.log_tokenizer(file, fill_value=-999)
    log2, cuts2 = L0.log_tokenizer(file2, fill_value=-999)
    log = log1[cuts1[0]:] + log2[:cuts2[0]]
    out = L0.L0_parser(msg, log, None, None, None, AOML_map, WMO_map, None, fill_value=-999)

    jsonschema.validate(out, schema)

    file_string = f"{AOML_map[float_id]:05}_{float_id:06}_L0_{_i:04}"
    outpath = f"examples/output/{folder_string}/{file_string}.json"
    with open(outpath, "w") as outfile:
        json.dump(out, outfile, indent=2)

examples/input/23031/23031.000
examples/input/23031/23031.001
examples/input/23031/23031.002
examples/input/23031/23031.003
examples/input/23031/23031.004
examples/input/23031/23031.005
examples/input/23031/23031.006
examples/input/23031/23031.007
examples/input/23031/23031.008
examples/input/23031/23031.009
examples/input/23031/23031.010
examples/input/23031/23031.011
examples/input/23031/23031.012
examples/input/23031/23031.013
examples/input/23031/23031.014
examples/input/23031/23031.015
examples/input/23031/23031.016
examples/input/23031/23031.017
[MSG] More than 1 "<EOT>" found
examples/input/23031/23031.018
examples/input/23031/23031.019
examples/input/23031/23031.020
examples/input/23031/23031.021
examples/input/23031/23031.022
examples/input/23031/23031.023
examples/input/23031/23031.024
examples/input/23031/23031.025
examples/input/23031/23031.026
examples/input/23031/23031.027
examples/input/23031/23031.028
[MSG] More than 1 "<EOT>" found
examples/input/23031/23031.029
exampl

## Core SBD float 23124 test (one profile)

In [15]:
float_id = 23124
cycle_id = 4

msg = L0.msg_tokenizer(f"examples/input/{float_id}/{float_id}.{cycle_id:03}")

In [16]:
out = L0.L0_parser(msg, None, None, None, None, AOML_map, WMO_map, None, consume=True, fill_value=-999)

In [17]:
with open("UWschema/UW_schema_L0.json", "r") as infile:
    schema = json.loads(infile.read())

In [18]:
jsonschema.validate(out, schema)

In [19]:
folder_string = f"{AOML_map[float_id]:05}"
file_string = f"{AOML_map[float_id]:05}_{float_id:06}_L0_{cycle_id:04}"
outpath = f"examples/output/{folder_string}/{file_string}.json"

if not os.path.isdir(f"examples/output/{folder_string}"):
    os.mkdir(f"examples/output/{folder_string}")

with open(outpath, "w") as outfile:
    json.dump(out, outfile, indent=2)

## BGC float 23808 test (one profile)

In [20]:
float_id = 23808
cycle_id = 4

in_path = f"examples/input/{float_id}/{float_id}.{cycle_id:03}"

msg = L0.msg_tokenizer(in_path)

log1, cuts1 = L0.log_tokenizer(f"examples/input/{float_id}/{float_id}.{cycle_id:03}", fill_value=-999)
log2, cuts2 = L0.log_tokenizer(f"examples/input/{float_id}/{float_id}.{cycle_id+1:03}", fill_value=-999)
log = log1[cuts1[0]:] + log2[:cuts2[0]]

dura = L0.dura_tokenizer(in_path, "MSC3")
isus = L0.isus_tokenizer(in_path)

In [21]:
out = L0.L0_parser(msg, log, dura, isus, None, AOML_map, WMO_map, OCR_map, consume=True)

In [22]:
with open("UWschema/UW_schema_L0.json", "r") as infile:
    schema = json.loads(infile.read())

In [23]:
jsonschema.validate(out, schema)

In [24]:
folder_string = f"{AOML_map[float_id]:05}"
file_string = f"{AOML_map[float_id]:05}_{float_id:06}_L0_{cycle_id:04}"
outpath = f"examples/output/{folder_string}/{file_string}.json"

if not os.path.isdir(f"examples/output/{folder_string}"):
    os.mkdir(f"examples/output/{folder_string}")

with open(outpath, "w") as outfile:
    json.dump(out, outfile, indent=2)

## BGC float 23808 test (loop over profiles)

In [25]:
with open("UWschema/UW_schema_L0.json", "r") as infile:
    schema = json.loads(infile.read())

In [26]:
float_id = 23808

folder_string = f"{AOML_map[float_id]:05}"
if not os.path.isdir(f"examples/output/{folder_string}"):
    os.mkdir(f"examples/output/{folder_string}")

for _i in range(29):
    file = f"examples/input/{float_id}/{float_id}.{_i:03}"
    file2 = f"examples/input/{float_id}/{float_id}.{_i+1:03}"
    print(file)
    msg = L0.msg_tokenizer(file)
    log1, cuts1 = L0.log_tokenizer(file, fill_value=-999)
    log2, cuts2 = L0.log_tokenizer(file2, fill_value=-999)
    log = log1[cuts1[0]:] + log2[:cuts2[0]]

    try:
        dura = L0.dura_tokenizer(file, "MSC3")
    except FileNotFoundError:
        dura = None

    try:
        isus = L0.isus_tokenizer(file)
    except FileNotFoundError:
        isus = None
    
    out = L0.L0_parser(msg, log, dura, isus, None, AOML_map, WMO_map, OCR_map, fill_value=-999)

    jsonschema.validate(out, schema)

    file_string = f"{AOML_map[float_id]:05}_{float_id:06}_L0_{_i:04}"
    outpath = f"examples/output/{folder_string}/{file_string}.json"
    with open(outpath, "w") as outfile:
        json.dump(out, outfile, indent=2)

examples/input/23808/23808.000
examples/input/23808/23808.001
examples/input/23808/23808.002
examples/input/23808/23808.003
[MSG] More than 1 "<EOT>" found
examples/input/23808/23808.004
examples/input/23808/23808.005
examples/input/23808/23808.006
examples/input/23808/23808.007
examples/input/23808/23808.008
[MSG] More than 1 "<EOT>" found
examples/input/23808/23808.009
[MSG] More than 1 "<EOT>" found
examples/input/23808/23808.010
examples/input/23808/23808.011
examples/input/23808/23808.012
examples/input/23808/23808.013
examples/input/23808/23808.014
examples/input/23808/23808.015
examples/input/23808/23808.016
examples/input/23808/23808.017
examples/input/23808/23808.018
examples/input/23808/23808.019
[MSG] More than 1 "<EOT>" found
examples/input/23808/23808.020
[DURA] More than 1 "<EOT>" found
[ISUS] More than 1 "<EOT>" found
examples/input/23808/23808.021
examples/input/23808/23808.022
examples/input/23808/23808.023
[MSG] More than 1 "<EOT>" found
examples/input/23808/23808.024